In [43]:
import pandas as pd

In [44]:
data = pd.read_csv('../fourth_downs_2018_2019.csv')

/tmp/ipykernel_5344/2015066739.py:1: DtypeWarning: Columns (0: lateral_receiver_player_id, 1: lateral_receiver_player_name, 2: lateral_punt_returner_player_id, 3: lateral_punt_returner_player_name, 4: qb_hit_2_player_id, 5: qb_hit_2_player_name, 6: pass_defense_2_player_id, 7: pass_defense_2_player_name, 8: fumbled_2_player_id, 9: fumbled_2_player_name, 10: fumbled_2_team, 11: fumble_recovery_2_team, 12: fumble_recovery_2_player_id, 13: fumble_recovery_2_player_name, 14: half_sack_1_player_id, 15: half_sack_1_player_name, 16: half_sack_2_player_id, 17: half_sack_2_player_name, 18: safety_player_name, 19: safety_player_id, 20: end_clock_time) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('../fourth_downs_2018_2019.csv')


In [45]:
# Columns of Interest
# 'home_team', 'away_team', 'posteam', 'defteam', 'td_team',
# 'penalty_team', 'return_team', 'fumbled_1_team', 'fumble_recovery_1_team',

In [46]:
for c in data.columns:
    print(c)

play_id
game_id
old_game_id
home_team
away_team
season_type
week
posteam
posteam_type
defteam
side_of_field
yardline_100
game_date
quarter_seconds_remaining
half_seconds_remaining
game_seconds_remaining
game_half
quarter_end
drive
sp
qtr
down
goal_to_go
time
yrdln
ydstogo
ydsnet
desc
play_type
yards_gained
shotgun
no_huddle
qb_dropback
qb_kneel
qb_spike
qb_scramble
pass_length
pass_location
air_yards
yards_after_catch
run_location
run_gap
field_goal_result
kick_distance
extra_point_result
two_point_conv_result
home_timeouts_remaining
away_timeouts_remaining
timeout
timeout_team
td_team
td_player_name
td_player_id
posteam_timeouts_remaining
defteam_timeouts_remaining
total_home_score
total_away_score
posteam_score
defteam_score
score_differential
posteam_score_post
defteam_score_post
score_differential_post
no_score_prob
opp_fg_prob
opp_safety_prob
opp_td_prob
fg_prob
safety_prob
td_prob
extra_point_prob
two_point_conversion_prob
ep
epa
total_home_epa
total_away_epa
total_home_rush_epa


In [47]:
## Select only the columns of interest to test functionality
columns_of_interest = ['posteam', 'defteam', 'home_team', 'away_team', 'touchdown', 'penalty', 'fumble', 'td_team',
                       'penalty_team', 'return_team', 'fumbled_1_team', 'fumble_recovery_1_team']
data_subset = data[columns_of_interest]
data_subset.head()

,posteam,defteam,home_team,away_team,touchdown,...,td_team,penalty_team,return_team,fumbled_1_team,fumble_recovery_1_team
0,ATL,PHI,PHI,ATL,0.0,...,NaN,NaN,NaN,NaN,NaN
1,PHI,ATL,PHI,ATL,0.0,...,NaN,PHI,ATL,NaN,NaN
2,ATL,PHI,PHI,ATL,0.0,...,NaN,NaN,NaN,NaN,NaN
3,PHI,ATL,PHI,ATL,0.0,...,NaN,NaN,ATL,NaN,NaN
4,ATL,PHI,PHI,ATL,0.0,...,NaN,NaN,PHI,NaN,NaN


In [48]:
## Create new columns: home_is_posteam, td_is_posteam, penalty_is_posteam, return_is_posteam, fumble_is_posteam as 0 or 1
data_subset['home_is_posteam'] = (data_subset['home_team'] == data_subset['posteam']).astype(int)
data_subset['td_is_posteam'] = (data_subset['td_team'] == data_subset['posteam']).astype(int)
data_subset['penalty_is_posteam'] = (data_subset['penalty_team'] == data_subset['posteam']).astype(int)
data_subset['return_is_posteam'] = (data_subset['return_team'] == data_subset['posteam']).astype(int)
data_subset['fumble_is_posteam'] = (data_subset['fumbled_1_team'] == data_subset['posteam']).astype(int)
data_subset.head()

,posteam,defteam,home_team,away_team,touchdown,...,home_is_posteam,td_is_posteam,penalty_is_posteam,return_is_posteam,fumble_is_posteam
0,ATL,PHI,PHI,ATL,0.0,...,0,0,0,0,0
1,PHI,ATL,PHI,ATL,0.0,...,1,0,1,0,0
2,ATL,PHI,PHI,ATL,0.0,...,0,0,0,0,0
3,PHI,ATL,PHI,ATL,0.0,...,1,0,0,0,0
4,ATL,PHI,PHI,ATL,0.0,...,0,0,0,0,0


In [49]:
## Drop the old irrelevant columns
data_subset = data_subset.drop(columns=['home_team', 'away_team', 'td_team',
                                       'penalty_team', 'return_team', 'fumbled_1_team', 'fumble_recovery_1_team'])
data_subset.head()

,posteam,defteam,touchdown,penalty,fumble,home_is_posteam,td_is_posteam,penalty_is_posteam,return_is_posteam,fumble_is_posteam
0,ATL,PHI,0.0,0.0,0.0,0,0,0,0,0
1,PHI,ATL,0.0,1.0,0.0,1,0,1,0,0
2,ATL,PHI,0.0,0.0,0.0,0,0,0,0,0
3,PHI,ATL,0.0,0.0,0.0,1,0,0,0,0
4,ATL,PHI,0.0,0.0,0.0,0,0,0,0,0


In [50]:
## Rewrite logic as a function that can be universally applied to any team column
def create_is_posteam_column(df, team_col):
    """
    This function takes in a dataframe and a team column name, creates a new column indicating whether the team in the specified column is the same as the posteam, and drops the old team column.
    Args:
        df (pd.DataFrame): The input dataframe.
        team_col (str): The name of the team column to compare with posteam.
    Returns:
        pd.DataFrame: The modified dataframe with the new is_posteam column and the old team column dropped.
    Example:
        df = create_is_posteam_column(df, 'home_team')
        Result would be df, with the new column 'home_is_posteam' and the old 'home_team' column dropped.
    """
    new_col_name = team_col.replace('_team', '_is_posteam')
    df[new_col_name] = (df[team_col] == df['posteam']).astype(int)
    ## Drop old team column
    df = df.drop(columns=[team_col])
    return df

In [51]:
## Test function on copy of original data
pd.set_option('display.max_columns', 10)  # Ensure all columns are displayed
data_test = data.copy()
data_test = create_is_posteam_column(data_test, 'home_team')
## Look at posteam, away_team, and home_is_posteam columns to verify functionality
data_test[['posteam', 'away_team', 'home_is_posteam']].head()

,posteam,away_team,home_is_posteam
0,ATL,ATL,0
1,PHI,ATL,1
2,ATL,ATL,0
3,PHI,ATL,1
4,ATL,ATL,0
